# Lab 05 — Model Comparison, Thresholds & Error Analysis

## Research question
Which baseline model fails least dangerously?

We will compare Logistic Regression, Random Forest, and XGBoost, then examine how the decision threshold changes the balance between missed attacks and false alarms.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

# If running from the notebooks/ folder locally:
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
# OPTIONAL — GOOGLE COLAB ONLY
# Run this cell if the UNSW-NB15 CSV files are not already in DATA_DIR.

try:
    from google.colab import files

    if not (DATA_DIR / "UNSW_NB15_training-set.csv").exists():
        print(
            "Upload UNSW_NB15_training-set.csv, "
            "UNSW_NB15_testing-set.csv, and optionally UNSW_NB15_features.csv"
        )
        uploaded = files.upload()
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        for filename, content in uploaded.items():
            (DATA_DIR / filename).write_bytes(content)

except ImportError:
    print("Not running in Colab. Put the dataset CSV files in:", DATA_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import (
    logistic_pipeline,
    random_forest_pipeline,
    xgboost_pipeline,
)
from src.evaluation import (
    binary_metrics,
    binary_metrics_frame,
    error_rows,
)

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [ ]:
models = {
    "Logistic Regression": logistic_pipeline(X_train),
    "Random Forest": random_forest_pipeline(X_train),
    "XGBoost": xgboost_pipeline(X_train),
}

metric_rows = []
model_scores = {}
model_predictions = {}

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)

    prediction = model.predict(X_test)
    score = model.predict_proba(X_test)[:, 1]

    model_predictions[name] = prediction
    model_scores[name] = score

    metric_rows.append(
        binary_metrics_frame(
            name,
            binary_metrics(y_test, prediction, score)
        )
    )

comparison = pd.concat(metric_rows)
display(comparison.sort_values("f1", ascending=False).round(4))

## Threshold experiment

The default threshold of 0.50 is not a law.

Lowering the threshold usually increases recall but also increases false positives.

Raising the threshold usually reduces alerts but can increase false negatives.

In [ ]:
chosen_model = "Random Forest"
score = model_scores[chosen_model]

rows = []
for threshold in [0.20, 0.35, 0.50, 0.65, 0.80]:
    prediction = (score >= threshold).astype(int)

    rows.append(
        binary_metrics_frame(
            f"{chosen_model} @ {threshold:.2f}",
            binary_metrics(y_test, prediction, score)
        )
    )

threshold_results = pd.concat(rows)
display(
    threshold_results[
        [
            "precision",
            "recall",
            "f1",
            "false_positive_rate",
            "false_negative_rate",
        ]
    ].round(4)
)

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, score)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(thresholds, precision[:-1], label="Precision")
ax.plot(thresholds, recall[:-1], label="Recall")
ax.set_title("Precision/Recall versus decision threshold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Metric")
ax.legend()
plt.show()

In [ ]:
errors = error_rows(
    X_test,
    y_test,
    model_predictions[chosen_model]
)

print(errors["error_type"].value_counts())
display(
    errors[errors["error_type"] == "false_negative"].head(10)
)

## Write your first research conclusion

Use this exact structure:

**Question:** Which model produced the most useful security baseline?

**Evidence:** Compare precision, recall, F1, false-positive rate, and false-negative rate.

**Interpretation:** Explain why the error tradeoff matters operationally.

**Limitation:** Explain why benchmark performance does not prove production effectiveness.

**Next experiment:** Propose a test involving class imbalance, concept drift, distribution shift, or adversarial perturbation.